[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/05-points-of-interest.ipynb)

# Points of Interest

`get_poi` discovers points of interest from OpenStreetMap within a search area. It supports category filtering, travel-time-aware search with Valhalla routing, and returns results sorted by distance or travel time.

In this notebook you will learn how to:

1. Search for POIs near a location
2. Inspect POI fields
3. Browse available category groups
4. Filter by category
5. Search multiple categories
6. Use travel-time-aware search
7. Compare walking vs driving travel times
8. Map POIs on a demographic choropleth
9. Import POIs from a CSV file

## Setup

In [ ]:
# Uncomment to install on Google Colab:
# !pip install 'socialmapper @ git+https://github.com/mihiarc/socialmapper.git'

from socialmapper import get_poi, import_poi_csv, create_isochrone, get_census_blocks, get_census_data, create_map
from IPython.display import Image, display

## 1. Basic POI Search

With no `travel_time`, `get_poi` searches within a 5 km radius and sorts by geodesic distance.

In [ ]:
pois = get_poi("Seattle, WA", limit=20)
print(f"POIs found: {len(pois)}")
for poi in pois[:5]:
    print(f"  {poi['name']:<40} {poi['category']:<20} {poi['distance_km']:.2f} km")

## 2. Inspect POI Fields

Each POI dict contains name, category, coordinates, distance, address, and OSM tags.

In [ ]:
sample_poi = pois[0]
for key, value in sample_poi.items():
    if key == "tags":
        print(f"{key}: <{len(value)} OSM tags>")
    else:
        print(f"{key}: {value}")

## 3. Available Category Groups

SocialMapper organizes OSM POIs into 10 high-level categories.

In [ ]:
from socialmapper.poi_categorization import POI_CATEGORY_MAPPING

for category, tags in POI_CATEGORY_MAPPING.items():
    print(f"{category:<20} ({len(tags)} tag values)")

## 4. Filter by Category

In [ ]:
restaurants = get_poi("Seattle, WA", categories=["food_and_drink"], limit=15)
print(f"Food & drink POIs: {len(restaurants)}")
for poi in restaurants[:5]:
    print(f"  {poi['name']:<40} {poi['distance_km']:.2f} km")

## 5. Multiple Categories

In [ ]:
edu_health = get_poi("Seattle, WA", categories=["education", "healthcare"], limit=20)
print(f"Education + Healthcare POIs: {len(edu_health)}")
for poi in edu_health[:8]:
    print(f"  {poi['name']:<40} {poi['category']:<15} {poi['distance_km']:.2f} km")

## 6. Travel-Time-Aware Search

When you pass `travel_time`, SocialMapper:
1. Creates an isochrone as the search boundary
2. Finds POIs inside that polygon
3. Computes actual routed travel times via Valhalla matrix API
4. Sorts results by `travel_time_minutes`

In [ ]:
pois_timed = get_poi(
    "Seattle, WA",
    categories=["shopping"],
    travel_time=10,
    travel_mode="drive",
    limit=15,
)

print(f"Shopping POIs within 10-min drive: {len(pois_timed)}")
for poi in pois_timed[:5]:
    travel = poi.get('travel_time_minutes', 'N/A')
    dist = poi.get('travel_distance_km', 'N/A')
    print(f"  {poi['name']:<35} {travel} min  /  {dist} km routed")

## 7. Walking vs Driving

Compare how many POIs are reachable within the same time budget by different modes.

In [ ]:
for mode in ["drive", "walk"]:
    result = get_poi(
        "Seattle, WA",
        categories=["food_and_drink"],
        travel_time=15,
        travel_mode=mode,
        limit=100,
    )
    print(f"{mode:>5}: {len(result)} food & drink POIs within 15 min")

## 8. Map POIs on a Demographic Choropleth

Combine everything: demographics as the choropleth, POIs as overlay points.

In [ ]:
# Build the demographic layer
iso = create_isochrone("Seattle, WA", travel_time=15, travel_mode="drive")
blocks = get_census_blocks(polygon=iso)
census = get_census_data(iso, variables=["population", "median_income"])

merged_blocks = []
for block in blocks:
    geoid = block["geoid"]
    if geoid in census.data:
        merged_blocks.append({**block, **census.data[geoid]})

# Get some food POIs for overlay
food_pois = get_poi("Seattle, WA", categories=["food_and_drink"], limit=30)
overlay = [{"lat": p["lat"], "lon": p["lon"], "name": p["name"]} for p in food_pois[:15]]

poi_map = create_map(
    data=merged_blocks,
    column="median_income",
    title="Median Income with Food & Drink POIs — Seattle",
    overlay_boundary=iso,
    overlay_points=overlay,
    show_stats=True,
)
display(Image(data=poi_map.image_data))

## 9. Import POIs from CSV

`import_poi_csv` reads a CSV with name, latitude, longitude, and type columns.

In [ ]:
import tempfile, os

csv_content = """name,latitude,longitude,type
Pike Place Market,47.6097,-122.3425,market
Space Needle,47.6205,-122.3493,landmark
University of Washington,47.6553,-122.3035,university
"""

with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False) as f:
    f.write(csv_content)
    csv_path = f.name

try:
    imported = import_poi_csv(csv_path)
    print(f"Imported {len(imported)} POIs:")
    for poi in imported:
        print(f"  {poi['name']:<30} ({poi['lat']}, {poi['lon']})")
finally:
    os.unlink(csv_path)

## Summary

| What you learned | API |
|---|---|
| Search POIs by radius | `get_poi(location, categories=[...])` |
| Travel-time search | `get_poi(location, travel_time=10)` → includes `travel_time_minutes` |
| Compare modes | `travel_mode='walk'` vs `'drive'` |
| Overlay on maps | `create_map(..., overlay_points=[...])` |
| Import from CSV | `import_poi_csv(path)` |

**Next notebook:** [06 — Multi-Location Comparison](06-multi-location-comparison.ipynb)